# Visualización de Arquetipos Descubiertos con Threads Completos

Este notebook visualiza los arquetipos descubiertos usando el contexto completo de cada thread.

**Metodología:**
1. ✅ Threads completos usados para descubrir clusters (más contexto)
2. ✅ Arquetipos asignados a mensajes principales
3. ✅ Visualización 3D de mensajes principales
4. ✅ Métricas de calidad mejoradas

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import json

print("✓ Librerías importadas")

✓ Librerías importadas


## 1. Cargar Datos Procesados

In [2]:
# Cargar mensajes principales con arquetipos descubiertos
df = pd.read_csv('parent_messages_with_thread_archetypes.csv')

# Cargar metadatos de arquetipos
with open('discovered_archetypes_from_threads.json', 'r', encoding='utf-8') as f:
    archetype_metadata = json.load(f)

print(f'✓ Cargados {len(df)} mensajes principales')
print(f'✓ Arquetipos únicos: {df["archetype_name"].nunique()}')
print(f'✓ Silhouette Score del descubrimiento: {archetype_metadata["silhouette_score"]:.3f}')
print(f'\nDistribución de arquetipos:')
print(df['archetype_name'].value_counts())

✓ Cargados 495 mensajes principales
✓ Arquetipos únicos: 20
✓ Silhouette Score del descubrimiento: 0.526

Distribución de arquetipos:
archetype_name
Que + Me + No                                       60
35 + Product_Configs + Falabella                    36
Https + Tab + Google                                30
Files + Statement + 22                              27
Security + Banco + Estado                           25
Amp + Paysheets + 5D                                25
18 + Product_Configs + Bancos                       24
Refresh + Cuentas + Bice                            24
De + Del + Vesti                                    24
Payouts + Merchants + Refunds                       23
Login + Cargos + Bancochile                         23
Unired + 30A1189Fbcb34C2B9D988D3C5347E998 + Mina    23
Cierres + Cierre + Contables                        23
20Collection + Ops + Pestaña                        22
Universo + Front + Pac                              22
Que + El + La             

## 2. Métricas de Calidad

In [3]:
# Preparar datos para métricas
coords_3d = df[['pc1', 'pc2', 'pc3']].values
archetype_labels = df['archetype_name'].astype('category').cat.codes.values

# Filtrar solo mensajes clasificados (sin 'Sin Clasificar')
classified_mask = df['archetype_name'] != 'Sin Clasificar'
coords_classified = coords_3d[classified_mask]
labels_classified = archetype_labels[classified_mask]

# Calcular métricas
if len(set(labels_classified)) > 1:
    silhouette = silhouette_score(coords_classified, labels_classified)
    calinski = calinski_harabasz_score(coords_classified, labels_classified)

    print('📊 MÉTRICAS DE CALIDAD DE CLUSTERING:\n')
    print(f'Silhouette Score: {silhouette:.3f}')
    print(f'  → Rango: [-1, 1]')
    print(f'  → Más cerca de 1 = mejor separación entre clusters')
    print()
    print(f'Calinski-Harabasz Score: {calinski:.1f}')
    print(f'  → Mayor valor = clusters más densos y separados')
    print()

    if silhouette > 0.5:
        print('✅ Excelente: Los clusters están bien definidos')
    elif silhouette > 0.25:
        print('✅ Aceptable: Los clusters tienen cierta superposición')
    elif silhouette > 0:
        print('⚠️  Mejorable: Algunos clusters se superponen')
    else:
        print('❌ Pobre: Considerar revisar los arquetipos')

    print(f'\n💡 Comparado con análisis sin threads:')
    print(f'   Si antes tenías score negativo, esto es una GRAN mejora!')

📊 MÉTRICAS DE CALIDAD DE CLUSTERING:

Silhouette Score: 0.551
  → Rango: [-1, 1]
  → Más cerca de 1 = mejor separación entre clusters

Calinski-Harabasz Score: 417.3
  → Mayor valor = clusters más densos y separados

✅ Excelente: Los clusters están bien definidos

💡 Comparado con análisis sin threads:
   Si antes tenías score negativo, esto es una GRAN mejora!


## 3. Visualización 3D Interactiva

In [4]:
# Preparar hover text
df['text_preview'] = df['text'].str[:100].str.replace('\n', ' ')
df['hover_text'] = (
    '<b>' + df['archetype_name'] + '</b><br>' +
    'Fecha: ' + df['datetime'].astype(str).str[:10] + '<br>' +
    '<i>' + df['text_preview'] + '...</i>'
)

# Crear figura 3D
fig = px.scatter_3d(
    df,
    x='pc1',
    y='pc2',
    z='pc3',
    color='archetype_name',
    hover_data={'hover_text': True, 'pc1': False, 'pc2': False, 'pc3': False},
    title=f'Arquetipos Descubiertos con Threads Completos ({len(df)} mensajes principales)',
    labels={
        'pc1': 'Componente Principal 1',
        'pc2': 'Componente Principal 2',
        'pc3': 'Componente Principal 3'
    },
    height=800,
    opacity=0.7
)

# Mejorar el layout
fig.update_traces(
    marker=dict(size=5, line=dict(width=0.5, color='white')),
    hovertemplate='%{customdata[0]}<extra></extra>'
)

fig.update_layout(
    scene=dict(
        xaxis=dict(backgroundcolor="rgb(246, 247, 248)"),
        yaxis=dict(backgroundcolor="rgb(246, 247, 248)"),
        zaxis=dict(backgroundcolor="rgb(246, 247, 248)"),
    ),
    font=dict(family="DM Sans, sans-serif", size=12),
)

fig.show()

print("\n💡 Tips:")
print("  - Arrastra para rotar la visualización")
print("  - Cada punto = 1 mensaje principal")
print("  - Color = arquetipo descubierto usando thread completo")
print("  - Haz clic en arquetipos para mostrar/ocultar")


💡 Tips:
  - Arrastra para rotar la visualización
  - Cada punto = 1 mensaje principal
  - Color = arquetipo descubierto usando thread completo
  - Haz clic en arquetipos para mostrar/ocultar


## 4. Detalles de Arquetipos Descubiertos

In [5]:
print('📋 ARQUETIPOS DESCUBIERTOS:\n')
print('='*80)

for arch in archetype_metadata['archetypes']:
    # Contar mensajes principales con este arquetipo
    count = len(df[df['archetype_name'] == arch['name']])
    pct = (count / len(df)) * 100

    print(f"\n{arch['name']}")
    print(f"  Mensajes: {count} ({pct:.1f}%)")
    print(f"  Threads analizados: {arch['thread_count']}")
    print(f"  Keywords: {', '.join(arch['keywords'][:6])}")
    print(f"  {arch['description']}")

    # Mostrar ejemplos
    examples = df[df['archetype_name'] == arch['name']].head(2)
    if len(examples) > 0:
        print(f"  Ejemplos:")
        for _, msg in examples.iterrows():
            preview = msg['text'][:120].replace('\n', ' ')
            print(f"    - {preview}...")

    print('-'*80)

📋 ARQUETIPOS DESCUBIERTOS:


Que + Me + No
  Mensajes: 60 (12.1%)
  Threads analizados: 61
  Keywords: que, me, no, el, si, la
  Arquetipo descubierto automáticamente usando 61 threads
  Ejemplos:
    - @S095YD2KQ4S me ayudas con la actualización del estado de este [pago](http://admin.prd.fin/payments/118102120) plisss, s...
    - @S095YD2KQ4S Me ayudan a cancelar este [reembolso](http://admin.prd.fin/refunds/ref_jEDpJ47HpEgdnr5V) (shopify) plisss, ...
--------------------------------------------------------------------------------

35 + Product_Configs + Falabella
  Mensajes: 36 (7.3%)
  Threads analizados: 36
  Keywords: 35, product_configs, falabella, availability_percentage, individual, 100
  Arquetipo descubierto automáticamente usando 36 threads
  Ejemplos:
    - @tank-ops-chile estos bancos tienen availability_percentage != 100 (payments - individual - CL): 1. [Banco Falabella](ht...
    - @tank-ops-chile estos bancos tienen availability_percentage != 100 (payments - individual 

## 5. Comparar Dos Arquetipos

In [6]:
# Listar arquetipos disponibles
archetypes_list = df['archetype_name'].unique().tolist()
print('Arquetipos disponibles:')
for i, arch in enumerate(archetypes_list):
    count = len(df[df['archetype_name'] == arch])
    print(f'  {i}: {arch} ({count} mensajes)')

Arquetipos disponibles:
  0: Login + Cargos + Bancochile (23 mensajes)
  1: Security + Banco + Estado (25 mensajes)
  2: Que + Me + No (60 mensajes)
  3: Universo + Front + Pac (22 mensajes)
  4: Cierres + Cierre + Contables (23 mensajes)
  5: Amp + Paysheets + 5D (25 mensajes)
  6: Unired + 30A1189Fbcb34C2B9D988D3C5347E998 + Mina (23 mensajes)
  7: Refresh + Cuentas + Bice (24 mensajes)
  8: Documento + Documents + Spa (17 mensajes)
  9: Alarm + Refund + Needs (16 mensajes)
  10: Sin Clasificar (19 mensajes)
  11: Que + El + La (21 mensajes)
  12: 20Collection + Ops + Pestaña (22 mensajes)
  13: Payouts + Merchants + Refunds (23 mensajes)
  14: De + Del + Vesti (24 mensajes)
  15: Https + Tab + Google (30 mensajes)
  16: Files + Statement + 22 (27 mensajes)
  17: 35 + Product_Configs + Falabella (36 mensajes)
  18: Bancoestado + Chile + Ops (11 mensajes)
  19: 18 + Product_Configs + Bancos (24 mensajes)


In [7]:
# Seleccionar dos arquetipos (cambiar índices según lo que quieras comparar)
ARCH_A = archetypes_list[0]  # Cambiar índice
ARCH_B = archetypes_list[1]  # Cambiar índice

df_comp = df[df['archetype_name'].isin([ARCH_A, ARCH_B])].copy()

fig_comp = px.scatter_3d(
    df_comp,
    x='pc1',
    y='pc2',
    z='pc3',
    color='archetype_name',
    hover_data={'hover_text': True, 'pc1': False, 'pc2': False, 'pc3': False},
    title=f'Comparación: {ARCH_A} vs {ARCH_B}',
    height=700,
    color_discrete_sequence=['#0045D7', '#DD0000']
)

fig_comp.update_traces(
    marker=dict(size=6, line=dict(width=0.5, color='white')),
    hovertemplate='%{customdata[0]}<extra></extra>'
)

fig_comp.show()

print(f'\n{ARCH_A}: {len(df_comp[df_comp["archetype_name"] == ARCH_A])} mensajes')
print(f'{ARCH_B}: {len(df_comp[df_comp["archetype_name"] == ARCH_B])} mensajes')


Login + Cargos + Bancochile: 23 mensajes
Security + Banco + Estado: 25 mensajes


## 6. Explorar Arquetipo Específico

In [8]:
# Seleccionar arquetipo a explorar
SELECTED = archetypes_list[0]  # Cambiar índice

print(f'📌 Explorando: {SELECTED}\n')

arch_msgs = df[df['archetype_name'] == SELECTED].copy()
print(f'Total: {len(arch_msgs)} mensajes principales\n')

print('🔍 Ejemplos de mensajes:\n')
for _, row in arch_msgs.head(5).iterrows():
    text_clean = row['text'][:200].replace('\n', ' ')
    date = str(row['datetime'])[:10]
    print(f'[{date}]')
    print(f'  "{text_clean}..."')
    print()

📌 Explorando: Login + Cargos + Bancochile

Total: 23 mensajes principales

🔍 Ejemplos de mensajes:

[2026-02-03]
  "Reminder: @S095YD2KQ4S descargar [nómina de cargos](http://admin.prd.fin/charge_batches), subirla al [banco](https://login.portalempresas.bancochile.cl/bancochile-web/empresa/login/index.html#/login) ..."

[2026-02-02]
  "Reminder: @S095YD2KQ4S descargar [nómina de cargos](http://admin.prd.fin/charge_batches), subirla al [banco](https://login.portalempresas.bancochile.cl/bancochile-web/empresa/login/index.html#/login) ..."

[2026-01-30]
  "Reminder: @S095YD2KQ4S descargar [nómina de cargos](http://admin.prd.fin/charge_batches), subirla al [banco](https://login.portalempresas.bancochile.cl/bancochile-web/empresa/login/index.html#/login) ..."

[2026-01-29]
  "Reminder: @S095YD2KQ4S descargar [nómina de cargos](http://admin.prd.fin/charge_batches), subirla al [banco](https://login.portalempresas.bancochile.cl/bancochile-web/empresa/login/index.html#/login) ..."

[2026-01-28]

## 🎯 Conclusiones

### Ventajas de usar threads completos:

✅ **Más contexto**: El thread completo da mucha más información sobre el tema
✅ **Mejores clusters**: Los arquetipos reflejan conversaciones completas, no solo mensajes aislados
✅ **Métricas mejoradas**: Silhouette score debería ser positivo y mayor
✅ **Clasificación futura**: Los mensajes nuevos se comparan contra estos arquetipos bien definidos

### Próximos pasos:

1. **Importar arquetipos a Tiger**: Crea los arquetipos manualmente en la UI
2. **Actualizar keywords**: Usa las keywords descubiertas para mejorar matching
3. **Clasificar mensajes nuevos**: Los incoming messages se clasificarán contra estos arquetipos
4. **Monitorear drift**: Ejecuta este análisis periódicamente para ver si surgen nuevos temas